# LeetCode 2080 — Range Frequency Queries bằng Python

Link đề: https://leetcode.com/problems/range-frequency-queries/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Design, Segment Tree

Skeleton LeetCode:

```python
class RangeFreqQuery:

    def __init__(self, arr: List[int]):
        ...

    def query(self, left: int, right: int, value: int) -> int:
        ...
```

Lộ trình:
1. **Đếm trực tiếp** trên đoạn mỗi truy vấn — `O(n)` mỗi lần.
2. **Prefix count cho mọi giá trị** — `O(1)` mỗi truy vấn nhưng bộ nhớ `O(n · V)`: không khả thi, hiểu để biết vì sao bỏ.
3. **Danh sách vị trí + 2 lần `bisect`** — `O(log n)` mỗi truy vấn, bộ nhớ `O(n)` (**nộp bài**).
4. **Heavy/light (căn bậc hai)** — giá trị xuất hiện nhiều dùng prefix, giá trị hiếm quét danh sách vị trí.
5. **Offline sweep** (khi biết trước mọi truy vấn): mỗi truy vấn = hiệu hai "đếm tới vị trí `x`".

## 1. Mục tiêu học

1. Thiết kế lớp có **tiền xử lý** ở `__init__` để mỗi `query` nhanh — tư duy "trả trước để truy vấn rẻ".
2. Nắm kỹ thuật **gom chỉ số theo giá trị** (`value → danh sách vị trí tăng dần`) và đếm phần tử trong `[left, right]` bằng `bisect_left` / `bisect_right`.
3. Phân biệt chính xác `bisect_left` và `bisect_right` — nguồn gốc của đa số lỗi off-by-one.
4. So sánh các phương án theo **bộ nhớ** chứ không chỉ thời gian (prefix theo mọi giá trị nổ bộ nhớ).
5. Viết **bộ chạy lệnh** mô phỏng LeetCode và đối chiếu với cài đặt brute force trên chuỗi lệnh ngẫu nhiên.

## 2. Đề bài & ví dụ

Thiết kế cấu trúc dữ liệu trả lời: **tần suất** của `value` trong đoạn con `arr[left..right]` (số lần `value` xuất hiện, tính cả hai đầu).

- `RangeFreqQuery(arr)`: khởi tạo với mảng `arr` (chỉ số từ 0).
- `query(left, right, value)`: trả về số chỉ số `i` với `left ≤ i ≤ right` và `arr[i] == value`.

**Ví dụ:**
```
["RangeFreqQuery", "query", "query"]
[[[12, 33, 4, 56, 22, 2, 34, 33, 22, 12, 34, 56]], [1, 2, 4], [0, 11, 33]]
→ [null, 1, 2]
```
`query(1, 2, 4)`: đoạn `[33, 4]` có một số `4`. `query(0, 11, 33)`: cả mảng có hai số `33`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ arr[i], value ≤ 10⁴`, `0 ≤ left ≤ right < n`, tối đa `10⁵` lần gọi `query`.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from collections import defaultdict, Counter
from typing import List

ARR = [12, 33, 4, 56, 22, 2, 34, 33, 22, 12, 34, 56]
example_cmds = ["RangeFreqQuery", "query", "query"]
example_args = [[ARR], [1, 2, 4], [0, 11, 33]]
example_out = [None, 1, 2]

def run_commands(cls, cmds, args):
    """Bộ chạy lệnh kiểu LeetCode: lệnh đầu tạo đối tượng, các lệnh sau gọi method."""
    out, obj = [], None
    for c, a in zip(cmds, args):
        if c == cmds[0]:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, c)(*a))
    return out

print(list(enumerate(ARR)))

[(0, 12), (1, 33), (2, 4), (3, 56), (4, 22), (5, 2), (6, 34), (7, 33), (8, 22), (9, 12), (10, 34), (11, 56)]


## 3. Kiến thức nền

### 3.1 `bisect_left` và `bisect_right` trên danh sách tăng dần

- `bisect_left(a, x)` = số phần tử **`< x`** (vị trí chèn bên trái các phần tử bằng `x`).
- `bisect_right(a, x)` = số phần tử **`≤ x`**.

Hệ quả: số phần tử trong `[lo, hi]` = `bisect_right(a, hi) - bisect_left(a, lo)`.

In [2]:
a = [1, 7, 7, 9, 12]
for x in (0, 7, 8, 12, 13):
    print(f"x = {x:>2}: bisect_left = {bisect_left(a, x)}, bisect_right = {bisect_right(a, x)}")
print("số phần tử trong [7, 9] =", bisect_right(a, 9) - bisect_left(a, 7))

x =  0: bisect_left = 0, bisect_right = 0
x =  7: bisect_left = 1, bisect_right = 3
x =  8: bisect_left = 3, bisect_right = 3
x = 12: bisect_left = 4, bisect_right = 5
x = 13: bisect_left = 5, bisect_right = 5
số phần tử trong [7, 9] = 3


### 3.2 `defaultdict(list)` để gom chỉ số theo giá trị

Duyệt `i` tăng dần và `append` ⇒ mỗi danh sách **tự động tăng dần** — không cần sắp xếp.

In [3]:
pos = defaultdict(list)
for i, x in enumerate(ARR):
    pos[x].append(i)
for v in sorted(pos):
    print(v, "->", pos[v])

2 -> [5]
4 -> [2]
12 -> [0, 9]
22 -> [4, 8]
33 -> [1, 7]
34 -> [6, 10]
56 -> [3, 11]


### 3.3 Prefix sum cho **một** giá trị

`pre[i]` = số lần `v` trong `arr[0..i-1]`; tần suất trên `[l, r]` = `pre[r + 1] - pre[l]`.

In [4]:
v = 33
pre = [0]
for x in ARR:
    pre.append(pre[-1] + (x == v))
print("pre[33] =", pre, "| tần suất 33 trên [0, 11] =", pre[12] - pre[0], "| trên [2, 6] =", pre[7] - pre[2])

pre[33] = [0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2] | tần suất 33 trên [0, 11] = 2 | trên [2, 6] = 0


### 3.4 Lớp Python và trạng thái

Mọi dữ liệu tiền xử lý lưu ở `self.*` trong `__init__`; `query` chỉ đọc. Vì `arr` **không đổi** sau khi khởi tạo, ta có thể tiền xử lý thoải mái.

## 4. Ý tưởng nền tảng / trực giác

Truy vấn `query(0, 11, 33)` hỏi: "trong các chỉ số `0..11`, có bao nhiêu chỉ số mang giá trị `33`?". Nếu đã có `pos[33] = [1, 7]` thì câu hỏi trở thành "trong danh sách `[1, 7]` có bao nhiêu số nằm trong `[0, 11]`?" — một bài **đếm trên mảng đã sắp xếp**, giải bằng hai lần tìm kiếm nhị phân.

Tính tay vài truy vấn:

| truy vấn | `pos[value]` | `bisect_left(pos, left)` | `bisect_right(pos, right)` | đáp án |
|---|---|---|---|---|
| `(1, 2, 4)` | `[2]` | 0 | 1 | 1 |
| `(0, 11, 33)` | `[1, 7]` | 0 | 2 | 2 |
| `(2, 8, 22)` | `[4, 8]` | 0 | 2 | 2 |
| `(5, 8, 12)` | `[0, 9]` | 1 | 1 | 0 |
| `(0, 11, 99)` | `[]` | 0 | 0 | 0 |

In [5]:
for left, right, value in [(1, 2, 4), (0, 11, 33), (2, 8, 22), (5, 8, 12), (0, 11, 99)]:
    p = pos.get(value, [])
    lo, hi = bisect_left(p, left), bisect_right(p, right)
    print(f"query({left}, {right}, {value}): pos = {p}, lo = {lo}, hi = {hi} -> {hi - lo}")

query(1, 2, 4): pos = [2], lo = 0, hi = 1 -> 1
query(0, 11, 33): pos = [1, 7], lo = 0, hi = 2 -> 2
query(2, 8, 22): pos = [4, 8], lo = 0, hi = 2 -> 2
query(5, 8, 12): pos = [0, 9], lo = 1, hi = 1 -> 0
query(0, 11, 99): pos = [], lo = 0, hi = 0 -> 0


## 5. Cách 1 — Brute Force / baseline: đếm trực tiếp

**Ý tưởng:** không tiền xử lý; mỗi truy vấn duyệt `arr[left..right]` và đếm.

In [6]:
class RangeFreqQueryBrute:
    def __init__(self, arr):
        self.arr = arr[:]

    def query(self, left, right, value):
        cnt = 0
        for i in range(left, right + 1):
            if self.arr[i] == value:
                cnt += 1
        return cnt

print(run_commands(RangeFreqQueryBrute, example_cmds, example_args), "kỳ vọng", example_out)

[None, 1, 2] kỳ vọng [None, 1, 2]


**Độ phức tạp:** `__init__` `O(n)`, `query` `O(right - left + 1) = O(n)`. Tệ nhất `10⁵` truy vấn × `10⁵` phần tử = `10¹⁰` — quá chậm (kể cả viết bằng `self.arr[left:right + 1].count(value)` chạy trong C cũng ≈ `10¹⁰` phép so sánh).

**Nút thắt:** mỗi truy vấn duyệt lại cả những phần tử **không phải** `value`. Ý tưởng: chỉ nhìn vào **vị trí của `value`**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tần suất của `value` trên `[left, right]` = số phần tử của `pos[value]` nằm trong `[left, right]`. Vì `pos[value]` tăng dần:

`query(left, right, value) = bisect_right(P, right) - bisect_left(P, left)` với `P = pos[value]`.

**Tổng kích thước** mọi danh sách `pos[·]` đúng bằng `n` ⇒ bộ nhớ `O(n)` — khác hẳn prefix theo mọi giá trị (`O(n · V)`).

**Quy tắc "nếu… thì…":**
- *Nếu* `value` chưa từng xuất hiện *thì* đáp án `0` (danh sách rỗng — hai `bisect` đều trả `0`).
- *Nếu* cần chỉ số bao gồm `right` *thì* dùng `bisect_right(P, right)` (hoặc `bisect_left(P, right + 1)` — tương đương với số nguyên).

In [7]:
# Hai cách viết đầu phải tương đương với chỉ số nguyên; và kiểm tra trên danh sách ngẫu nhiên
rng = random.Random(9)
for _ in range(2000):
    P = sorted(rng.sample(range(40), rng.randint(0, 15)))
    l = rng.randrange(40); r = rng.randrange(l, 40)
    a = bisect_right(P, r) - bisect_left(P, l)
    b = bisect_left(P, r + 1) - bisect_left(P, l)
    assert a == b == sum(1 for x in P if l <= x <= r)
print("bisect_right(P, r) == bisect_left(P, r + 1) với số nguyên; cả hai đếm đúng [l, r]")

bisect_right(P, r) == bisect_left(P, r + 1) với số nguyên; cả hai đếm đúng [l, r]


Lưu ý: tương đương `bisect_right(P, r) == bisect_left(P, r + 1)` chỉ đúng vì chỉ số là **số nguyên** (không có giá trị nào nằm giữa `r` và `r + 1`). Với số thực, chỉ dùng `bisect_right`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix count cho mọi giá trị (và vì sao bỏ)

Lưu `pre[v]` cho mọi `v` xuất hiện. Truy vấn `O(1)`. Bộ nhớ: `D` giá trị phân biệt × `(n + 1)`.

In [8]:
class RangeFreqQueryPrefixAll:
    def __init__(self, arr):
        self.pre = {}
        for v in set(arr):
            p = [0]
            for x in arr:
                p.append(p[-1] + (x == v))
            self.pre[v] = p

    def query(self, left, right, value):
        p = self.pre.get(value)
        return 0 if p is None else p[right + 1] - p[left]

print(run_commands(RangeFreqQueryPrefixAll, example_cmds, example_args), "kỳ vọng", example_out)
n, D = 10**5, 10**4
print(f"tệ nhất: {D} giá trị × {n + 1} ô ≈ {D * (n + 1):.2e} số nguyên — hàng GB bộ nhớ, dựng mất cả phút")

[None, 1, 2] kỳ vọng [None, 1, 2]
tệ nhất: 10000 giá trị × 100001 ô ≈ 1.00e+09 số nguyên — hàng GB bộ nhớ, dựng mất cả phút


**Độ phức tạp:** `__init__` `O(n · D)`, `query` `O(1)`, Space `O(n · D)`. Chỉ hợp lý khi `D` rất nhỏ (ví dụ giá trị ≤ 100 như bài 1906).

### 7.2 Cách 3 — Danh sách vị trí + bisect (code nộp)

In [9]:
class RangeFreqQueryBisect:
    def __init__(self, arr):
        self.pos = defaultdict(list)
        for i, x in enumerate(arr):
            self.pos[x].append(i)

    def query(self, left, right, value):
        p = self.pos.get(value)
        if not p:
            return 0
        return bisect_right(p, right) - bisect_left(p, left)

print(run_commands(RangeFreqQueryBisect, example_cmds, example_args), "kỳ vọng", example_out)

[None, 1, 2] kỳ vọng [None, 1, 2]


Dùng `.get(value)` thay vì `self.pos[value]`: với `defaultdict`, truy cập bằng `[]` **tạo khóa mới** cho mỗi giá trị chưa có — không sai kết quả, nhưng làm từ điển phình to khi có nhiều truy vấn giá trị lạ.

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `__init__` `O(n)`, `query` `O(log f)` với `f` = tần suất của `value` (`≤ n`). Space `O(n)`.

### 7.3 Cách 4 — Heavy / light (căn bậc hai)

Chọn ngưỡng `B ≈ √n`. Giá trị **nặng** (xuất hiện `> B` lần — có tối đa `n / B` giá trị như vậy) dùng prefix (`O(1)` truy vấn, tổng bộ nhớ `O(n · n / B) = O(n √n)`). Giá trị **nhẹ** quét tuyến tính danh sách vị trí (`≤ B` phần tử).

In [10]:
class RangeFreqQueryHeavyLight:
    def __init__(self, arr, B=None):
        n = len(arr)
        self.B = B or max(1, int(n ** 0.5))
        self.pos = defaultdict(list)
        for i, x in enumerate(arr):
            self.pos[x].append(i)
        self.pre = {}
        for v, p in self.pos.items():
            if len(p) > self.B:                     # giá trị nặng
                pr = [0] * (n + 1)
                for i in p:
                    pr[i + 1] = 1
                for i in range(n):
                    pr[i + 1] += pr[i]
                self.pre[v] = pr

    def query(self, left, right, value):
        pr = self.pre.get(value)
        if pr is not None:
            return pr[right + 1] - pr[left]
        return sum(1 for i in self.pos.get(value, ()) if left <= i <= right)

print(run_commands(RangeFreqQueryHeavyLight, example_cmds, example_args), "kỳ vọng", example_out)

[None, 1, 2] kỳ vọng [None, 1, 2]


**Độ phức tạp:** `__init__` `O(n √n)` (tối đa `√n` giá trị nặng × `n`), `query` `O(√n)` tệ nhất. Space `O(n √n)`. Không tốt hơn Cách 3 ở bài này — nhưng là **khung** hữu ích khi thay "đếm" bằng thao tác không bisect được.

### 7.4 Cách 5 — Offline sweep (biết trước toàn bộ truy vấn)

`query(l, r, v) = C(r, v) - C(l - 1, v)` với `C(x, v)` = số lần `v` trong `arr[0..x]`. Sắp xếp các "điểm hỏi" `(x, v, dấu, id)` theo `x`, quét mảng từ trái sang phải với một `Counter`.

In [11]:
def offline_answers(arr, queries):
    events = []                                # (x, value, sign, qid)
    for qid, (l, r, v) in enumerate(queries):
        events.append((r, v, +1, qid))
        if l > 0:
            events.append((l - 1, v, -1, qid))
    events.sort()
    ans = [0] * len(queries)
    cnt = Counter(); e = 0
    for i, x in enumerate(arr):
        cnt[x] += 1
        while e < len(events) and events[e][0] == i:
            _, v, s, qid = events[e]
            ans[qid] += s * cnt[v]
            e += 1
    return ans

print(offline_answers(ARR, [(1, 2, 4), (0, 11, 33), (2, 8, 22), (5, 8, 12), (0, 11, 99)]))

[1, 2, 2, 0, 0]


**Độ phức tạp:** `O((n + q) + q log q)` (sắp xếp sự kiện), Space `O(n + q)`. Không dùng được trực tiếp cho bài thiết kế **online** (truy vấn đến từng cái), nhưng là kỹ thuật chuẩn cho dạng "trả lời hàng loạt".

## 8. TRACE từng bước

Trace Cách 3: in danh sách vị trí, hai vị trí `bisect` và các phần tử được đếm.

In [12]:
def trace_query(obj, left, right, value):
    p = obj.pos.get(value, [])
    lo = bisect_left(p, left); hi = bisect_right(p, right)
    marks = "".join("^" if lo <= k < hi else "." for k in range(len(p)))
    print(f"query({left:>2}, {right:>2}, {value:>2}) | pos = {p} | lo = {lo} (số vị trí < {left}) | "
          f"hi = {hi} (số vị trí ≤ {right}) | được đếm: {p[lo:hi]} -> {hi - lo}")

obj = RangeFreqQueryBisect(ARR)
for q in [(1, 2, 4), (0, 11, 33), (2, 8, 22), (8, 8, 22), (9, 11, 56), (0, 0, 33)]:
    trace_query(obj, *q)

query( 1,  2,  4) | pos = [2] | lo = 0 (số vị trí < 1) | hi = 1 (số vị trí ≤ 2) | được đếm: [2] -> 1
query( 0, 11, 33) | pos = [1, 7] | lo = 0 (số vị trí < 0) | hi = 2 (số vị trí ≤ 11) | được đếm: [1, 7] -> 2
query( 2,  8, 22) | pos = [4, 8] | lo = 0 (số vị trí < 2) | hi = 2 (số vị trí ≤ 8) | được đếm: [4, 8] -> 2
query( 8,  8, 22) | pos = [4, 8] | lo = 1 (số vị trí < 8) | hi = 2 (số vị trí ≤ 8) | được đếm: [8] -> 1
query( 9, 11, 56) | pos = [3, 11] | lo = 1 (số vị trí < 9) | hi = 2 (số vị trí ≤ 11) | được đếm: [11] -> 1
query( 0,  0, 33) | pos = [1, 7] | lo = 0 (số vị trí < 0) | hi = 0 (số vị trí ≤ 0) | được đếm: [] -> 0


In [13]:
def trace_offline(arr, queries):
    events = []
    for qid, (l, r, v) in enumerate(queries):
        events.append((r, v, +1, qid))
        if l > 0: events.append((l - 1, v, -1, qid))
    events.sort(); print("sự kiện (x, value, dấu, qid):", events)
    ans = [0] * len(queries); cnt = Counter(); e = 0
    for i, x in enumerate(arr):
        cnt[x] += 1
        while e < len(events) and events[e][0] == i:
            _, v, s, qid = events[e]; ans[qid] += s * cnt[v]
            print(f"  tại i = {i}: C({i}, {v}) = {cnt[v]}, dấu {s:+d} -> ans[{qid}] = {ans[qid]}"); e += 1
    return ans

print(trace_offline(ARR, [(1, 2, 4), (2, 8, 22), (5, 8, 12)]))

sự kiện (x, value, dấu, qid): [(0, 4, -1, 0), (1, 22, -1, 1), (2, 4, 1, 0), (4, 12, -1, 2), (8, 12, 1, 2), (8, 22, 1, 1)]
  tại i = 0: C(0, 4) = 0, dấu -1 -> ans[0] = 0
  tại i = 1: C(1, 22) = 0, dấu -1 -> ans[1] = 0
  tại i = 2: C(2, 4) = 1, dấu +1 -> ans[0] = 1
  tại i = 4: C(4, 12) = 1, dấu -1 -> ans[2] = -1
  tại i = 8: C(8, 12) = 1, dấu +1 -> ans[2] = 0
  tại i = 8: C(8, 22) = 2, dấu +1 -> ans[1] = 2
[1, 2, 0]


## 9. Code nộp LeetCode

In [14]:
class RangeFreqQuery:

    def __init__(self, arr: List[int]):
        self.pos = defaultdict(list)
        for i, x in enumerate(arr):
            self.pos[x].append(i)          # i tăng dần ⇒ mỗi danh sách đã sắp xếp

    def query(self, left: int, right: int, value: int) -> int:
        p = self.pos.get(value)
        if not p:
            return 0
        return bisect_right(p, right) - bisect_left(p, left)

got = run_commands(RangeFreqQuery, example_cmds, example_args)
print(got, "OK" if got == example_out else "SAI")

[None, 1, 2] OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Sau `__init__`, `pos[v]` là danh sách **tăng ngặt** gồm đúng các chỉ số `i` có `arr[i] = v`. (Duyệt `i = 0, 1, …, n-1` và `append` vào danh sách của `arr[i]`.)

**Bổ đề 2.** Với danh sách tăng `P`: `bisect_left(P, left)` = `|{p ∈ P : p < left}|` và `bisect_right(P, right)` = `|{p ∈ P : p ≤ right}|`. Vì `left ≤ right`, tập `{p < left}` ⊆ `{p ≤ right}` nên hiệu = `|{p ∈ P : left ≤ p ≤ right}|`.

Kết hợp hai bổ đề: hiệu = số chỉ số `i ∈ [left, right]` với `arr[i] = value` — đúng định nghĩa tần suất. ✓

**Cách 4.** Với giá trị nặng, `pr` là prefix của chỉ báo `arr[i] == v` (đặt `1` tại `i + 1` rồi cộng dồn) ⇒ `pr[r + 1] - pr[l]` đúng. Giá trị nhẹ đếm trực tiếp. ✓

**Cách 5.** `C(r, v) - C(l - 1, v)` = số lần `v` trong `arr[l..r]` (với `C(-1, v) = 0`, vì vậy bỏ sự kiện khi `l = 0`). Sự kiện tại `x` được xử lý **ngay sau** khi đã cộng `arr[x]` vào `cnt` ⇒ `cnt[v] = C(x, v)`. ✓

## 11. Phân tích Time / Space

`n = len(arr) ≤ 10⁵`, `q ≤ 10⁵` truy vấn, `D` = số giá trị phân biệt (`≤ 10⁴`), `f` = tần suất của giá trị được hỏi.

| Cách | `__init__` | `query` | Space |
|---|---|---|---|
| 1. Đếm trực tiếp | `O(n)` | `O(n)` | `O(n)` |
| 2. Prefix mọi giá trị | `O(n D)` | `O(1)` | `O(n D)` |
| 3. Vị trí + bisect | `O(n)` | `O(log f)` | `O(n)` |
| 4. Heavy/light | `O(n √n)` | `O(√n)` | `O(n √n)` |
| 5. Offline sweep | — | tổng `O(n + q log q)` | `O(n + q)` |

Đo trên `n = 10⁵`, `q = 10⁵` (giá trị ngẫu nhiên `1..10⁴` và trường hợp lệch: một nửa mảng là cùng một giá trị):

In [15]:
rng = random.Random(1)
n, q = 10**5, 10**5
arr_rand = [rng.randint(1, 10**4) for _ in range(n)]
arr_skew = [7 if rng.random() < 0.5 else rng.randint(1, 10**4) for _ in range(n)]
qs = []
for _ in range(q):
    l = rng.randrange(n); r = rng.randrange(l, n)
    qs.append((l, r, rng.choice((7, rng.randint(1, 10**4)))))

for aname, arr in [("random", arr_rand), ("skewed", arr_skew)]:
    res = {}
    for name, cls in [("bisect", RangeFreqQueryBisect), ("heavy/light", RangeFreqQueryHeavyLight), ("submit", RangeFreqQuery)]:
        t0 = time.perf_counter(); obj = cls(arr); t1 = time.perf_counter()
        res[name] = [obj.query(*x) for x in qs]; t2 = time.perf_counter()
        print(f"{aname:6s} {name:11s}: init {t1 - t0:.2f}s, {q} queries {t2 - t1:.2f}s")
    t0 = time.perf_counter(); res["offline"] = offline_answers(arr, qs)
    print(f"{aname:6s} offline    : tổng {time.perf_counter() - t0:.2f}s")
    assert len({tuple(v) for v in res.values()}) == 1

obj = RangeFreqQueryBrute(arr_rand); t0 = time.perf_counter()
for x in qs[:200]: obj.query(*x)
print(f"brute: chỉ 200 truy vấn mất {time.perf_counter() - t0:.2f}s (≈ {(time.perf_counter() - t0) * 500:.0f}s cho 1e5)")

random bisect     : init 0.01s, 100000 queries 0.04s
random heavy/light: init 0.02s, 100000 queries 0.07s
random submit     : init 0.01s, 100000 queries 0.03s


random offline    : tổng 0.23s
skewed bisect     : init 0.01s, 100000 queries 0.06s
skewed heavy/light: init 0.02s, 100000 queries 0.06s


skewed submit     : init 0.01s, 100000 queries 0.06s
skewed offline    : tổng 0.23s


brute: chỉ 200 truy vấn mất 0.16s (≈ 79s cho 1e5)


## 12. Bộ test

Đối chiếu mọi cách với brute force trên **chuỗi lệnh ngẫu nhiên** (giá trị nhỏ để có nhiều trùng lặp; có cả `value` không xuất hiện).

In [16]:
classes = [RangeFreqQueryPrefixAll, RangeFreqQueryBisect, RangeFreqQueryHeavyLight, RangeFreqQuery]

for cls in classes + [RangeFreqQueryBrute]:
    assert run_commands(cls, example_cmds, example_args) == example_out

edge_cases = [
    ([5], [(0, 0, 5), (0, 0, 4)], [1, 0]),                         # n = 1
    ([1, 1, 1, 1], [(0, 3, 1), (1, 2, 1), (2, 2, 1)], [4, 2, 1]),  # toàn một giá trị
    ([1, 2, 1, 2], [(1, 1, 1), (0, 0, 1), (3, 3, 2)], [0, 1, 1]),  # đoạn một phần tử
    ([3, 1, 3], [(0, 2, 2), (0, 2, 10**4)], [0, 0]),               # giá trị không có
]
for arr, qs_e, want in edge_cases:
    for cls in classes:
        obj = cls(arr)
        assert [obj.query(*x) for x in qs_e] == want, (cls, arr)

rng = random.Random(2024)
for _ in range(400):
    n = rng.randint(1, 30); V = rng.choice([1, 3, 8, 30])
    arr = [rng.randint(1, V) for _ in range(n)]
    cmds, args = ["RangeFreqQuery"], [[arr]]
    for _ in range(rng.randint(1, 20)):
        l = rng.randrange(n); r = rng.randrange(l, n)
        cmds.append("query"); args.append([l, r, rng.randint(1, V + 1)])
    want = run_commands(RangeFreqQueryBrute, cmds, args)
    for cls in classes:
        assert run_commands(cls, cmds, args) == want, (cls, arr, args)
    assert offline_answers(arr, [tuple(a) for a in args[1:]]) == want[1:]
print("ví dụ + edge + 400 chuỗi lệnh ngẫu nhiên: mọi cách khớp brute force")

ví dụ + edge + 400 chuỗi lệnh ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `bisect_left(p, right)` cho đầu phải (bỏ sót `right`)

In [17]:
class WrongRightEnd(RangeFreqQueryBisect):
    def query(self, left, right, value):
        p = self.pos.get(value, [])
        return bisect_left(p, right) - bisect_left(p, left)      # SAI: không đếm vị trí == right

obj_w, obj_ok = WrongRightEnd(ARR), RangeFreqQuery(ARR)
for x in [(1, 2, 4), (0, 11, 33), (0, 7, 33), (2, 2, 4)]:
    print(f"query{x}: sai = {obj_w.query(*x)} | đúng = {obj_ok.query(*x)}")

query(1, 2, 4): sai = 0 | đúng = 1
query(0, 11, 33): sai = 2 | đúng = 2
query(0, 7, 33): sai = 1 | đúng = 2
query(2, 2, 4): sai = 0 | đúng = 1


`bisect_left(p, right)` đếm vị trí `< right`. Ví dụ `query(1, 2, 4)`: `4` ở đúng chỉ số `2 = right` bị bỏ sót. **Sửa:** `bisect_right(p, right)` hoặc `bisect_left(p, right + 1)`.

### Lỗi 2 — Dùng `bisect_right(p, left)` cho đầu trái (bỏ sót `left`)

In [18]:
class WrongLeftEnd(RangeFreqQueryBisect):
    def query(self, left, right, value):
        p = self.pos.get(value, [])
        return bisect_right(p, right) - bisect_right(p, left)    # SAI: trừ cả vị trí == left

obj_w = WrongLeftEnd(ARR)
for x in [(1, 2, 4), (1, 11, 33), (2, 2, 4), (0, 11, 12)]:
    print(f"query{x}: sai = {obj_w.query(*x)} | đúng = {obj_ok.query(*x)}")

query(1, 2, 4): sai = 1 | đúng = 1
query(1, 11, 33): sai = 1 | đúng = 2
query(2, 2, 4): sai = 0 | đúng = 1
query(0, 11, 12): sai = 1 | đúng = 2


**Sửa:** đầu trái dùng `bisect_left(p, left)` (đếm số vị trí `< left`).

### Lỗi 3 — Cắt lát thiếu một phần tử: `arr[left:right].count(value)`

In [19]:
class WrongSlice(RangeFreqQueryBrute):
    def query(self, left, right, value):
        return self.arr[left:right].count(value)                 # SAI: lát cắt không gồm right

obj_w = WrongSlice(ARR)
print(run_commands(WrongSlice, example_cmds, example_args), "| đúng =", example_out)

[None, 0, 2] | đúng = [None, 1, 2]


Lát cắt Python `a[l:r]` **không** gồm `r`. **Sửa:** `arr[left:right + 1]` (và dù đúng, cách này vẫn `O(n)` mỗi truy vấn).

### Lỗi 4 — Tạo danh sách vị trí **không tăng dần** rồi vẫn `bisect`

In [20]:
class WrongOrder(RangeFreqQueryBisect):
    def __init__(self, arr):
        self.pos = defaultdict(list)
        for i in range(len(arr) - 1, -1, -1):                     # SAI: duyệt ngược ⇒ danh sách giảm dần
            self.pos[arr[i]].append(i)

obj_w = WrongOrder(ARR)
for x in [(1, 2, 4), (0, 11, 33), (0, 5, 12), (5, 11, 34)]:
    print(f"query{x}: sai = {obj_w.query(*x)} | đúng = {obj_ok.query(*x)}")

query(1, 2, 4): sai = 1 | đúng = 1
query(0, 11, 33): sai = 2 | đúng = 2
query(0, 5, 12): sai = 2 | đúng = 1
query(5, 11, 34): sai = 2 | đúng = 2


`bisect` **giả định** danh sách tăng dần; với danh sách giảm, kết quả vô nghĩa. **Sửa:** duyệt `i` tăng dần (hoặc `sort()` mỗi danh sách).

### Lỗi 5 — Truy cập `self.pos[value]` với `defaultdict` cho mọi truy vấn

In [21]:
obj = RangeFreqQueryBisect([1, 2, 3])
class LeakyQuery(RangeFreqQueryBisect):
    def query(self, left, right, value):
        p = self.pos[value]                                       # tạo khóa mới nếu chưa có
        return bisect_right(p, right) - bisect_left(p, left)

leaky = LeakyQuery([1, 2, 3])
for v in range(1000, 6000):
    leaky.query(0, 2, v); obj.query(0, 2, v)
print("số khóa: leaky =", len(leaky.pos), "| dùng .get =", len(obj.pos))

số khóa: leaky = 5003 | dùng .get = 3


Kết quả vẫn đúng, nhưng mỗi giá trị lạ tạo một danh sách rỗng mới — với `10⁵` truy vấn có thể thêm `10⁴` khóa. **Sửa:** dùng `.get(value)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số lần `x` xuất hiện trong đoạn `[l, r]`, mảng tĩnh, nhiều truy vấn" ⇒ **value → danh sách vị trí** + hai `bisect`.
- Miền giá trị **rất nhỏ** (≤ 26, ≤ 100) ⇒ prefix count theo giá trị cũng được (`O(1)` truy vấn).
- Đếm trong `[lo, hi]` trên mảng đã sắp xếp: `bisect_right(a, hi) - bisect_left(a, lo)` — thuộc lòng.
- Biết trước mọi truy vấn ⇒ cân nhắc **offline**: tách truy vấn đoạn thành hiệu hai truy vấn tiền tố.
- Có **cập nhật** phần tử ⇒ thay list bằng `SortedList` (Bài 5) hoặc dùng cây Fenwick theo từng giá trị.
- Họ bài: 1906 *Minimum Absolute Difference Queries*, 1157 *Online Majority Element In Subarray*, 2055 *Plates Between Candles*, 2389 *Longest Subsequence With Limited Sum*.

## 15. Biến thể của bài

1. **Vị trí đầu tiên / cuối cùng của `value` trong `[l, r]`** (Bài 4): `bisect_left` cho đầu tiên, `bisect_right - 1` cho cuối cùng.
2. **Có cập nhật** `update(i, val)`: mỗi giá trị giữ `SortedList` vị trí; cập nhật `O(log n)` (Bài 5).
3. **Đếm số phần tử có giá trị trong `[a, b]`** trên đoạn `[l, r]`: cần merge-sort tree / wavelet tree / offline Fenwick.
4. **1157 Online Majority Element:** tìm phần tử chiếm `≥ threshold` trong đoạn — kết hợp ngẫu nhiên hóa/Boyer-Moore với chính kỹ thuật đếm tần suất của bài này để xác minh ứng viên.

In [22]:
class MajorityChecker:
    """Biến thể 4 (LeetCode 1157) — kiểm tra ứng viên bằng tần suất (đếm bằng bisect)."""
    def __init__(self, arr, seed=0):
        self.arr = arr; self.rq = RangeFreqQuery(arr); self.rng = random.Random(seed)
    def query(self, left, right, threshold):
        for _ in range(20):                                  # xác suất trượt ≤ 2^-20
            x = self.arr[self.rng.randint(left, right)]
            if self.rq.query(left, right, x) >= threshold:
                return x
        return -1

mc = MajorityChecker([1, 1, 2, 2, 1, 1])
print([mc.query(0, 5, 4), mc.query(0, 3, 3), mc.query(2, 3, 2)], "kỳ vọng [1, -1, 2]")

[1, -1, 2] kỳ vọng [1, -1, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | `query` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm trực tiếp | duyệt đoạn | `O(n)` | `O(n)` | kiểm chứng |
| 2. Prefix mọi giá trị | `pre[v][r+1] - pre[v][l]` | `O(1)` | `O(n D)` | `D` rất nhỏ |
| 3. Vị trí + bisect | đếm trên list tăng | `O(log n)` | `O(n)` | **nộp bài** |
| 4. Heavy/light | ngưỡng `√n` | `O(√n)` | `O(n √n)` | thao tác không bisect được |
| 5. Offline | hiệu hai tiền tố | `O(1)` khấu hao | `O(n + q)` | biết trước truy vấn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** với `ARR` của ví dụ, tính `query(3, 11, 56)`, `query(0, 4, 22)`, `query(6, 10, 34)`.

**Bài 2 (tính tay):** `pos[12] = [0, 9]`. Tính `bisect_left(pos, 1)`, `bisect_right(pos, 9)` và đáp án `query(1, 9, 12)`.

**Bài 3 (điền TODO):** hoàn thiện `count_in_range(sorted_list, lo, hi)` trả về số phần tử trong `[lo, hi]`.

**Bài 4 (tự viết lại):** viết lớp `RangeFirstLast` với `first(left, right, value)` và `last(left, right, value)` trả về chỉ số đầu tiên / cuối cùng của `value` trong `[left, right]`, hoặc `-1`.

**Bài 5 (mini challenge):** thêm method `update(index, value)` gán `arr[index] = value`. Thiết kế lớp `RangeFreqQueryUpdatable` với cả `update` và `query` đều `O(log n)` (gợi ý: `SortedList`). Đối chiếu brute force trên chuỗi lệnh ngẫu nhiên. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [23]:
def count_in_range(sorted_list, lo, hi):
    # TODO: dùng bisect_right / bisect_left
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `pos[56] = [3, 11]`, `pos[22] = [4, 8]`, `pos[34] = [6, 10]`. (2) Đếm các vị trí rơi vào đoạn. (3) Kết quả lần lượt `2`, `1`, `2`.

**Bài 2:** (1) Số vị trí `< 1` là `1` (chỉ số `0`). (2) Số vị trí `≤ 9` là `2`. (3) `2 - 1 = 1`.

**Bài 3:** (1) Một dòng. (2) `bisect_right(a, hi) - bisect_left(a, lo)`. (3) Nếu `lo > hi` có thể ra số âm — trả `max(0, …)`.

**Bài 4:** (1) `k = bisect_left(p, left)`; nếu `k < len(p)` và `p[k] ≤ right` thì `p[k]`. (2) `k = bisect_right(p, right) - 1`; nếu `k ≥ 0` và `p[k] ≥ left` thì `p[k]`. (3) Kiểm chứng bằng duyệt tuyến tính.

**Bài 5:** (1) `self.pos[v] = SortedList(...)`. (2) `update`: xóa `index` khỏi `pos[arr[index]]`, thêm vào `pos[value]`, gán `arr[index]`. (3) `SortedList.bisect_left/bisect_right` dùng như `bisect`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
ok = RangeFreqQuery(ARR)
assert [ok.query(3, 11, 56), ok.query(0, 4, 22), ok.query(6, 10, 34)] == [2, 1, 2]

# Bài 2
p12 = [0, 9]
assert bisect_left(p12, 1) == 1 and bisect_right(p12, 9) == 2 and ok.query(1, 9, 12) == 1

# Bài 3
def count_in_range_ans(a, lo, hi):
    return max(0, bisect_right(a, hi) - bisect_left(a, lo))

rng = random.Random(31)
for _ in range(500):
    a = sorted(rng.randint(0, 20) for _ in range(rng.randint(0, 10)))
    lo, hi = rng.randint(-2, 22), rng.randint(-2, 22)
    assert count_in_range_ans(a, lo, hi) == sum(1 for x in a if lo <= x <= hi)

# Bài 4
class RangeFirstLast:
    def __init__(self, arr):
        self.pos = defaultdict(list)
        for i, x in enumerate(arr): self.pos[x].append(i)
    def first(self, left, right, value):
        p = self.pos.get(value, []); k = bisect_left(p, left)
        return p[k] if k < len(p) and p[k] <= right else -1
    def last(self, left, right, value):
        p = self.pos.get(value, []); k = bisect_right(p, right) - 1
        return p[k] if k >= 0 and p[k] >= left else -1

for _ in range(300):
    arr = [rng.randint(1, 4) for _ in range(rng.randint(1, 15))]
    fl = RangeFirstLast(arr)
    l = rng.randrange(len(arr)); r = rng.randrange(l, len(arr)); v = rng.randint(1, 5)
    idx = [i for i in range(l, r + 1) if arr[i] == v]
    assert fl.first(l, r, v) == (idx[0] if idx else -1) and fl.last(l, r, v) == (idx[-1] if idx else -1)
print("first/last ví dụ:", RangeFirstLast(ARR).first(0, 11, 33), RangeFirstLast(ARR).last(0, 11, 33))
print("Bài 1-4: OK")

first/last ví dụ: 1 7
Bài 1-4: OK


In [25]:
# Bài 5 — có cập nhật, dùng SortedList
from sortedcontainers import SortedList

class RangeFreqQueryUpdatable:
    def __init__(self, arr):
        self.arr = arr[:]
        self.pos = defaultdict(SortedList)
        for i, x in enumerate(arr):
            self.pos[x].add(i)
    def update(self, index, value):
        old = self.arr[index]
        if old == value:
            return None
        self.pos[old].remove(index)
        self.pos[value].add(index)
        self.arr[index] = value
        return None
    def query(self, left, right, value):
        p = self.pos.get(value)
        return 0 if not p else p.bisect_right(right) - p.bisect_left(left)

class UpdatableBrute:
    def __init__(self, arr): self.arr = arr[:]
    def update(self, index, value): self.arr[index] = value
    def query(self, left, right, value): return self.arr[left:right + 1].count(value)

rng = random.Random(55)
for _ in range(300):
    n = rng.randint(1, 20); arr = [rng.randint(1, 4) for _ in range(n)]
    cmds, args = ["Init"], [[arr]]
    for _ in range(rng.randint(1, 30)):
        if rng.random() < 0.4:
            cmds.append("update"); args.append([rng.randrange(n), rng.randint(1, 4)])
        else:
            l = rng.randrange(n); cmds.append("query"); args.append([l, rng.randrange(l, n), rng.randint(1, 5)])
    assert run_commands(RangeFreqQueryUpdatable, cmds, args) == run_commands(UpdatableBrute, cmds, args)

demo_c = ["Init", "query", "update", "query", "query"]
demo_a = [[ARR], [0, 11, 33], [2, 33], [0, 11, 33], [1, 2, 4]]
print("Bài 5 ví dụ:", run_commands(RangeFreqQueryUpdatable, demo_c, demo_a), "| brute:", run_commands(UpdatableBrute, demo_c, demo_a))

Bài 5 ví dụ: [None, 2, None, 3, 0] | brute: [None, 2, None, 3, 0]


## 20. Template nhớ nhanh + Checklist

```python
# Tần suất của value trong arr[l..r], mảng tĩnh
pos = defaultdict(list)
for i, x in enumerate(arr):
    pos[x].append(i)                     # tăng dần sẵn

def query(l, r, value):
    p = pos.get(value)
    if not p: return 0
    return bisect_right(p, r) - bisect_left(p, l)   # số vị trí trong [l, r]
```

**Checklist trước khi submit:**
- [ ] Đầu phải dùng `bisect_right(p, right)` (hoặc `bisect_left(p, right + 1)`).
- [ ] Đầu trái dùng `bisect_left(p, left)`.
- [ ] Danh sách vị trí **tăng dần** (duyệt `i` từ 0).
- [ ] Giá trị không xuất hiện ⇒ `0`; dùng `.get` để khỏi tạo khóa rác.
- [ ] Không dựng prefix cho mọi giá trị (bộ nhớ `O(n · D)`).